# 10-2 Advanced Machine Learning: Model Zoo, Ensembles, Evaluation Suite, and SHAP

The first notebook (`10_ml_baseline`) honestly showed you: **on the 280-row Legionnaires' disease dataset, Random Forest can barely beat logistic regression** -- real-world outbreak prediction is humbling. So when does ML actually "win handily"? The answer: **when disease risk is non-linear, has interactions, and there's enough data.**

This notebook steps onto a bigger stage and walks you through a complete ML workflow:

- **Model zoo**: Decision Tree, Random Forest, XGBoost, LASSO -- each paired with a clinical metaphor
- **Ensembles**: bagging / boosting / **stacking (Super Learner)**
- **Evaluation suite**: ROC-AUC, PR-AUC, sensitivity/specificity/PPV/NPV, calibration
- **SHAP**: explaining a black-box model to a clinician
- **Class imbalance, overfitting, and "ML is a tool, not a replacement for epidemiological judgment"**

> 🏭 **Changing stages: why a "synthetic sandbox"?**
> Legionnaires' disease only has 280 rows and weak signal, so it can't show off what ML can do. Instead, we switch to a synthetic **"regional respiratory-outbreak notification roll-up dataset"** (imagine a CDC AI office aggregating notifications from many facilities, n≈2500). It's a **teaching sandbox**, deliberately designed with a "U-shaped age risk" and an "immunosuppressed × exposure interaction" -- exactly the kind of signal that logistic regression can't catch but ML can.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
# --- Packages ---
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                             confusion_matrix, f1_score, roc_curve, precision_recall_curve)
from sklearn.calibration import calibration_curve
import xgboost as xgb
import shap

from epi_learning.viz import configure_chinese_font
configure_chinese_font()

## Step 1 -- Building a Teaching Sandbox (Synthetic Data)

We synthesize a dataset of 2,500 people, where `severe` = whether the case progressed to severe illness. Two signals that "only ML can catch" are deliberately baked in:

- **U-shaped age risk**: both very young and very old people are high-risk (a linear `age` term can't capture this)
- **Immunosuppressed × exposure interaction**: risk only spikes when someone is **both** immunosuppressed **and** highly exposed

Two **pure noise** columns (`noise_lab`, `noise_ward`) are also thrown in, to see whether the model gets fooled.

In [ ]:
def make_cohort(seed=10, n=2500):
    rng = np.random.default_rng(seed)
    age            = rng.integers(20, 90, n)
    immunosuppressed = rng.binomial(1, 0.20, n)
    exposure       = rng.uniform(0, 1, n)          # exposure dose (aerosol/shower)
    vaccinated     = rng.binomial(1, 0.45, n)
    diabetes       = rng.binomial(1, 0.25, n)
    copd           = rng.binomial(1, 0.15, n)
    sex            = rng.binomial(1, 0.5, n)
    noise_lab      = rng.normal(0, 1, n)            # pure noise
    noise_ward     = rng.integers(0, 5, n)          # pure noise

    age_u = ((age - 55) / 20) ** 2                  # <- U-shaped: both young and old are high
    logit = (-3.4
             + 1.6 * age_u                          # nonlinear
             + 2.6 * immunosuppressed * exposure    # <- interaction
             + 1.8 * exposure + 1.1 * diabetes + 1.0 * copd
             - 1.3 * vaccinated + 0.4 * sex)
    severe = rng.binomial(1, 1 / (1 + np.exp(-logit)))

    X = pd.DataFrame({"age": age, "immunosuppressed": immunosuppressed, "exposure": exposure.round(3),
                      "vaccinated": vaccinated, "diabetes": diabetes, "copd": copd, "sex": sex,
                      "noise_lab": noise_lab.round(2), "noise_ward": noise_ward})
    return X, pd.Series(severe, name="severe")

X, y = make_cohort()
print(f"n = {len(X)}, severe proportion = {y.mean():.1%}")
X.head()

## Step 2 -- The Three-Way Split: train / validation / test (the most important foundation)

Ch07 taught you "never peek at the future"; the ML version of that rule is the **three-way split**:

| Dataset | Proportion | Role | Plain-language analogy |
|---|---|---|---|
| **train** | 60% | Learn the formula (fit the model) | Attend class, do homework |
| **validation** | 20% | Tune hyperparameters, pick a model | Mock exam (you can check the answers afterward) |
| **test** | 20% | Final evaluation, **opened only once** | Final exam (once you've seen it, it's spent) |

![train/val/test three-way split](../images/train_val_test_split_en.svg)

> 🚨 **Data leakage is ML's #1 killer**: if any information from the test set sneaks into the training process, the model will ace its own exam and then fail badly in production. Three cardinal sins: ① standardizing/SMOTE-ing *before* splitting (must be done **inside** each fold); ② using "part of the outcome" as a feature (e.g., predicting infection from symptoms); ③ using information from the future.
>
> ⏳ **Time-series / spatial data need extra care**: time series must use `TimeSeriesSplit` (never shuffle randomly); spatial data needs spatial CV (don't split up neighboring regions) -- otherwise it's still peeking.

In [ ]:
# Stratified three-way split: first split 70/30, then split the 30 in half -> val 15% / test 15% (test_size=0.5 here makes val ≈ test)
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.3, random_state=10, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=10, stratify=y_tmp)
print(f"train={len(X_train)}  validation={len(X_val)}  test={len(X_test)}")
print(f"All three splits have similar severe proportions: {y_train.mean():.2f} / {y_val.mean():.2f} / {y_test.mean():.2f} (thanks to stratify)")

## Step 3 -- The Model Zoo (each with a clinical metaphor)

- 🩺 **Decision Tree = an ER triage flowchart**: a chain of yes/no questions (Fever? >65? Immunosuppressed?) that walks down to a high-risk/low-risk box. Very easy to explain, but the questioning is rigid -- swap in a different batch of patients and the whole tree can grow crooked (unstable, prone to overfitting).
- 👥 **Random Forest = a multi-specialty consult vote (bagging)**: gather a few hundred doctors, each of whom only sees **part** of the chart and **part** of the tests, draws their own tree, and casts one vote; majority rules. No single doctor knows everything, but a crowd of "slightly different, independently-reasoning" doctors voting together is more stable than any one doctor alone.
- 📈 **XGBoost = an error-book cram school (boosting)**: the first tutor finishes teaching, then hands the "questions still gotten wrong" to a second tutor who specializes in fixing exactly those, a third tutor mops up what's left... each round focuses on correcting the previous round's mistakes. Powerful, but prone to over-correcting.
- 🧳 **LASSO (L1 logistic regression) = packing with a weight limit**: the L1 penalty acts like a luggage weight limit, forcing unimportant variables' coefficients to **zero** and keeping only a handful of truly useful factors → a lean, report-ready list. That's exactly why epidemiologists love using it as a baseline.

In [ ]:
zoo = {
    "LASSO":        LogisticRegression(penalty="l1", solver="liblinear", C=0.5,
                                       class_weight="balanced", max_iter=2000),
    "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=1, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=250, max_depth=7, random_state=1,
                                            class_weight="balanced"),
    "XGBoost":       xgb.XGBClassifier(n_estimators=250, max_depth=4, learning_rate=0.07,
                                       eval_metric="logloss", random_state=1),
}

fitted, rows = {}, []
for name, model in zoo.items():
    model.fit(X_train, y_train)
    fitted[name] = model
    p = model.predict_proba(X_test)[:, 1]
    rows.append({"Model": name, "AUC": roc_auc_score(y_test, p),
                 "PR-AUC": average_precision_score(y_test, p),
                 "Brier": brier_score_loss(y_test, p)})
zoo_scores = pd.DataFrame(rows).round(3)
print(zoo_scores.to_string(index=False))
print("\n→ LASSO (linear) AUC is only 0.71; tree-based models hit 0.84+ -- this is exactly where ML wins when there's 'nonlinearity + interaction'")

## Step 4 -- Ensembles: three ways to "pool the wisdom of the crowd"

- **Bagging** (Random Forest): a group of models look at portions of the data **in parallel** and vote/average → reduces variance, more stable.
- **Boosting** (XGBoost): models work **in relay**, each one specializing in the previous one's residuals → reduces bias, more accurate.
- 🎯 **Stacking = a mission-control commander (Super Learner)**: the tree, forest, XGBoost, and LASSO each hand over a probability, and the **commander (the meta-model, usually logistic regression) never looks at the patient directly** -- instead it learns "which expert to trust more, and when," and combines them with weights into a final call. This is the **Super Learner** (van der Laan) from the epidemiology literature.

In [ ]:
stack = StackingClassifier(
    estimators=[(k, v) for k, v in zoo.items()],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5,
)
stack.fit(X_train, y_train)
p_stack = stack.predict_proba(X_test)[:, 1]
print(f"Stacking (Super Learner) test-set AUC = {roc_auc_score(y_test, p_stack):.3f}")
print("→ Stacking usually ≥ the best single model, and is less likely to bet on the wrong horse (reduces reliance on any one model's bias)")

## Step 5 -- The Evaluation Suite: don't look at just one number

Different tasks call for different metrics. One-line summary: during **screening**, prioritize "don't miss anyone" (sensitivity, NPV, PR-AUC); during **confirmation / resource allocation**, prioritize "don't cry wolf, and get the probabilities right" (specificity, PPV, calibration).

| Metric | When to use it | Plain-language epi meaning |
|---|---|---|
| **ROC-AUC** | Model selection, across all thresholds | The probability a case ranks above a healthy person; overly optimistic when classes are imbalanced |
| **PR-AUC** | **Positives are rare** (severe cases/deaths) | Focuses on "how real are the positives you caught" -- more honest than AUC |
| **Sensitivity** | Screening, when missing a case is costly | Of the true cases, what fraction did you catch |
| **Specificity** | When false alarms are costly | Of the truly healthy, what fraction were correctly cleared |
| **PPV** | Bedside decisions right now (heavily affected by prevalence) | "The model said positive -- what's the chance they're really sick" -- what clinicians care about most |
| **Calibration / Brier** | When you need to use the probability as a number (bed allocation, risk communication) | Of the group predicted 70%, do about 70% actually get sick? **Good ranking ≠ accurate probability** |

In [ ]:
p_rf = fitted["Random Forest"].predict_proba(X_test)[:, 1]

# Confusion matrix @ threshold 0.5 -> epi metrics
tn, fp, fn, tp = confusion_matrix(y_test, (p_rf >= 0.5)).ravel()
sens, spec = tp / (tp + fn), tn / (tn + fp)
ppv, npv = tp / (tp + fp), tn / (tn + fn)
print(f"Random Forest @0.5: sensitivity={sens:.2f} specificity={spec:.2f} PPV={ppv:.2f} NPV={npv:.2f} F1={f1_score(y_test,(p_rf>=0.5)):.2f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
# ROC
fpr, tpr, _ = roc_curve(y_test, p_rf)
axes[0].plot(fpr, tpr, color="#D97757"); axes[0].plot([0,1],[0,1],"--",color="#6B6B6B")
axes[0].set_title(f"ROC curve (AUC={roc_auc_score(y_test,p_rf):.2f})"); axes[0].set_xlabel("1 - Specificity"); axes[0].set_ylabel("Sensitivity")
# PR
prec, rec, _ = precision_recall_curve(y_test, p_rf)
axes[1].plot(rec, prec, color="#6A9BCC"); axes[1].axhline(y_test.mean(), ls="--", color="#6B6B6B")
axes[1].set_title(f"PR curve (PR-AUC={average_precision_score(y_test,p_rf):.2f})"); axes[1].set_xlabel("Recall/Sensitivity"); axes[1].set_ylabel("Precision/PPV")
# Calibration
frac_pos, mean_pred = calibration_curve(y_test, p_rf, n_bins=5)
axes[2].plot(mean_pred, frac_pos, "o-", color="#788C5D"); axes[2].plot([0,1],[0,1],"--",color="#6B6B6B")
axes[2].set_title("Calibration (predicted probability vs. observed rate)"); axes[2].set_xlabel("Predicted probability"); axes[2].set_ylabel("Observed fraction")
plt.tight_layout(); plt.show()
print("→ The closer the calibration curve hugs the diagonal, the more 'predicted 70%' really means 'about 70% actually occurred' -- essential to check before using probabilities for bed allocation")

## Step 6 -- Class Imbalance: the accuracy trap

In a real outbreak, "severe/death" is often the minority class. When that happens, **accuracy can lie to you**.

In [ ]:
# Downsample the severe class to ~8%, to create an imbalanced dataset
rng = np.random.default_rng(3)
pos_idx = y[y == 1].index
keep = rng.choice(pos_idx, size=int(len(pos_idx) * 0.12), replace=False)
imb_idx = y[y == 0].index.union(pd.Index(keep))
y_imb = y.loc[imb_idx]
print(f"Imbalanced data: severe cases are only {y_imb.mean():.1%} of the sample")
print(f"👉 Just guess 'never severe' for everyone, and accuracy is {1 - y_imb.mean():.1%} -- alarmingly high, but completely useless (it misses every single patient)!")
print("→ So when classes are imbalanced: ① don't look at accuracy, look at PR-AUC / sensitivity instead;")
print("  ② use class_weight='balanced' (or apply SMOTE only inside the train fold, otherwise it leaks) to upweight the minority class")

## Step 7 -- SHAP: Explaining the Black Box to a Clinician

> 💰 **The year-end bonus-splitting metaphor**: SHAP uses Shapley values from game theory, asking "how much worse would the prediction be without this feature?" It averages each feature's marginal contribution -- **with vs. without** -- across every possible order of adding features. That lets it tell a **single patient's** story: they were flagged high-risk because "immunosuppressed +0.3, exposure +0.2, age 80 +0.15" -- exactly the language a clinician needs to make sense of a black box.

In [ ]:
explainer = shap.TreeExplainer(fitted["XGBoost"])
shap_values = explainer.shap_values(X_test)

# 1) Feature importance (beeswarm): which features matter most overall
shap.summary_plot(shap_values, X_test, show=False)
plt.tight_layout(); plt.show()

In [ ]:
# 2) Dependence plot: SHAP uncovers the 'U-shaped age risk' we baked in -- SHAP values curve upward for both young and old
shap.dependence_plot("age", shap_values, X_test, interaction_index=None, show=False)
plt.tight_layout(); plt.show()
print("→ SHAP successfully recovered the 'U-shaped age risk' we deliberately designed in -- something a logistic model's linear age term can never draw")

## Step 8 -- Overfitting: aces the training exam, fails in production

When a model is too complex and the data too limited, it ends up **memorizing the noise in the training set** -- the training score looks stellar, but the test score is dismal.

In [ ]:
deep_tree = DecisionTreeClassifier(max_depth=None, random_state=1).fit(X_train, y_train)  # unlimited depth -> overfitting
auc_tr = roc_auc_score(y_train, deep_tree.predict_proba(X_train)[:, 1])
auc_te = roc_auc_score(y_test,  deep_tree.predict_proba(X_test)[:, 1])
print(f"Unlimited-depth decision tree: train AUC = {auc_tr:.3f}, test AUC = {auc_te:.3f}")
print(f"→ Near-perfect training, collapsed test performance = textbook overfitting. It even memorized noise like noise_lab/noise_ward")
print("  Countermeasures: limit depth / pruning, regularization (LASSO's L1), early stopping, more data, cross-validation")

## Wrap-up: ML Is a "Tool," Not a Replacement for Epidemiological Judgment

You've walked through a complete ML workflow. But keep these points carved into memory -- they're what separates an **epidemiologist** using ML from an engineer using it:

1. **Good ranking ≠ accurate probability**: a high AUC doesn't mean you can use the model to allocate beds -- check **calibration**.
2. **Important ≠ causal**: SHAP saying `exposure` is important doesn't mean "changing exposure will prevent disease"; feature importance is **not an intervention target** (causal inference is what Ch12 is for).
3. **External validation**: a model trained at Pine and Cypress Nursing Home might collapse elsewhere (temporal/spatial dataset shift) -- you must re-validate on **new data**.
4. **Fairness**: is the subgroup AUC consistent across sex and age groups? Might the model be especially inaccurate for some group?
5. **Where the data comes from determines what the model learns**: notification bias and selection bias will be faithfully learned -- and amplified -- by the model.

> 🧭 **One-liner**: ML helps you **find patterns in a pile of variables and make predictions**; but **why** that pattern exists, **how** to intervene on it, and whether it **generalizes** -- that's always an epidemiologist's judgment call. **A model is a telescope, not a steering wheel.**